# Saving Results with `save_results()`

VESIcal provides a general-purpose `save_results()` function that can save any combination of VESIcal objects (Samples, Calculate results, BatchFiles), pandas DataFrames, dictionaries, and scalars to CSV or Excel files. This function supports flexible output modes for organizing your data.

First, import VESIcal:

In [ ]:
import sys
sys.path.insert(0, '../')

import VESIcal as v

## Set up some example data

Let's create a Sample and run a calculation so we have objects to save.

In [ ]:
mysample = v.Sample({
    'SiO2': 77.5,
    'TiO2': 0.08,
    'Al2O3': 12.5,
    'Fe2O3': 0.207,
    'Cr2O3': 0.0,
    'FeO': 0.473,
    'MnO': 0.0,
    'MgO': 0.03,
    'NiO': 0.0,
    'CoO': 0.0,
    'CaO': 0.43,
    'Na2O': 3.98,
    'K2O': 4.88,
    'P2O5': 0.0,
    'H2O': 5.5,
    'CO2': 0.05
})

satP = v.calculate_saturation_pressure(sample=mysample, temperature=1200, model="IaconoMarziano")
diss = v.calculate_dissolved_volatiles(sample=mysample, pressure=1000, temperature=1200, X_fluid=1, model="IaconoMarziano")

## Save a single object to CSV

The simplest usage: pass a filename and a single object. By default, results are saved as CSV.

In [ ]:
v.save_results("satP_result.csv", satP)

## Save a list of mixed types

`save_results()` accepts lists containing any mix of supported types. All objects are converted to DataFrames and combined.

In [ ]:
v.save_results("mixed_results.csv", [mysample, satP, {"note": "example data"}])

## Save to Excel

Use `filetype="excel"` to save as an `.xlsx` file instead of CSV.

In [ ]:
v.save_results("results.xlsx", [mysample, satP], filetype="excel")

## Output modes

The `mode` parameter controls how multiple objects are organized:

- **`single_sheet`** (default): All data combined into one file/sheet
- **`multi_sheet`**: Each object on its own Excel sheet (Excel only)
- **`multi_file`**: Each object saved to a separate file

In [ ]:
# All data on one sheet
v.save_results("single.xlsx", [mysample, satP, diss],
               filetype="excel", mode="single_sheet")

In [ ]:
# Each item on its own sheet in one Excel file
v.save_results("multi_sheet.xlsx", [mysample, satP, diss],
               filetype="excel", mode="multi_sheet")

In [ ]:
# Each item in its own CSV file (creates out_1.csv, out_2.csv, out_3.csv)
v.save_results("out.csv", [mysample, satP, diss],
               filetype="csv", mode="multi_file")

## Adding descriptions

Use the `descriptions` parameter to label each object with metadata. This adds a `Description` column to the output.

In [ ]:
v.save_results("described.csv", [mysample, satP, diss],
               descriptions=["Sample composition", "Saturation pressure", "Dissolved volatiles"])

## Working with a BatchFile

BatchFile objects are also supported. Here we load a file and save batch calculation results.

In [ ]:
myfile = v.BatchFile('../../manuscript/example_data.xlsx')
batch_satP = myfile.calculate_saturation_pressure(temperature=1000)

v.save_results("batch_results.xlsx", [myfile, batch_satP],
               filetype="excel", mode="multi_sheet",
               descriptions=["Input compositions", "Saturation pressures"])

## Cleanup

Remove the example files generated by this notebook.

In [ ]:
import os
for f in ["satP_result.csv", "mixed_results.csv", "results.xlsx",
          "single.xlsx", "multi_sheet.xlsx",
          "out_1.csv", "out_2.csv", "out_3.csv",
          "described.csv", "batch_results.xlsx"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")